In [0]:
catalog = "flights_project"
schema_bronze = "bronze"
volume_path = f"/Volumes/{catalog}/{schema_bronze}/raw_landing"
checkpoint_path = f"/Volumes/{catalog}/{schema_bronze}/raw_landing/_checkpoints/flights_bronze"

display(dbutils.fs.ls(volume_path))

path,name,size,modificationTime
dbfs:/Volumes/flights_project/bronze/raw_landing/flights_2025_01.csv,flights_2025_01.csv,116286638,1786905963000
dbfs:/Volumes/flights_project/bronze/raw_landing/flights_2025_04.csv,flights_2025_04.csv,126417350,1786905963000
dbfs:/Volumes/flights_project/bronze/raw_landing/flights_2025_07.csv,flights_2025_07.csv,137573374,1786905963000
dbfs:/Volumes/flights_project/bronze/raw_landing/flights_2025_10.csv,flights_2025_10.csv,132483941,1786905963000


In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", checkpoint_path)
      .option("header", "true")
      .option("cloudFiles.inferColumnTypes", "true")
      .load(volume_path))

from pyspark.sql.functions import current_timestamp, col

df_bronze = (df
             .withColumn("_ingested_at", current_timestamp())
             .withColumn("_source_file", col("_metadata.file_path")))

(df_bronze.writeStream
 .format("delta")
 .option("checkpointLocation", checkpoint_path)
 .trigger(availableNow=True)
 .toTable(f"{catalog}.{schema_bronze}.flights_raw"))

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM flights_project.bronze.flights_raw;

SELECT 
  MONTH(to_date(FL_DATE, 'M/d/yyyy h:mm:ss a')) AS flight_month, 
  COUNT(*) AS cnt
FROM flights_project.bronze.flights_raw
GROUP BY MONTH(to_date(FL_DATE, 'M/d/yyyy h:mm:ss a'))
ORDER BY flight_month;t

flight_month,cnt
1,539747
4,583950
7,631428
10,605844
